In [21]:
import requests
import pandas as pd
import json
import logging
import numpy as np
from xgboost import XGBClassifier


api_url = "http://13.62.6.52:9881/datahub/market_data"
default_timeout = 30
default_limit = 1000

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

In [22]:
def fetch_market_data(
        start_date: str,
        end_date: str,
        market: str = None,
        market_area: str = None,
        limit: int = default_limit
        ):
    """Retrieve market data from REST API

    Args:
        start_date (str): start of requested delivery period
        end_date (str): end of requested delivery period
        market (str, optional): specified market to filter on. Defaults to None.
        market_area (str, optional): specified market area to filter on. Defaults to None.
        limit (int, optional): maximum number of rows to request. Defaults to default_limit.

    Returns:
        pd.DataFrame: raw market data returned by the API 
    """

    params_dict = {
        "start_date": start_date,
        "end_date": end_date,
        "limit": limit
    }

    logger.info("Requesting market data from %s to %s", start_date, end_date)

    try:
        response = requests.get(
            url=api_url,
            params=params_dict,
            timeout=default_timeout
        )
        payload = response.json()
        rows = payload.get("rows")
        raw_df = pd.DataFrame.from_records(rows)

    except requests.RequestException as reqex:
        logger.error("API request failed: %s", reqex)
        raise


    return raw_df

In [ ]:
def fetch_market_history(
        start_date: str,
        end_date: str,
        chunk_hours: int = 6
        ):
    """
    Retrieve a complete market-data history in smaller chunks to avoid
    the API row limit.

    Args:
        start_date(str): start timestamp in ISO-8601 format.

        end_date (str): end timestamp in ISO-8601 format.

        chunk_hours (int): number of hours requested per API call.

    Returns:
        pd.DataFrame : combined raw market data across the requested period.
    """

    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    chunks = []

    current_start = start

    while current_start < end:

        current_end = min(
            current_start + pd.Timedelta(hours=chunk_hours),
            end
        )

        chunk = fetch_market_data(
            start_date=current_start.isoformat(),
            end_date=current_end.isoformat()
        )

        chunks.append(chunk)

        current_start = current_end

    if not chunks:
        return pd.DataFrame()

    data = pd.concat(
        chunks,
        ignore_index=True
    )

    # Chunk boundaries may be returned by both neighbouring requests
    data = data.drop_duplicates().reset_index(drop=True)

    return data

In [24]:
def clean_market_data(
        raw_df:pd.DataFrame
        ):
    """Tidy up and clean the raw dataframe into appropriate dtypes

    Args:
        raw_df (pd.DataFrame): original data from the API request before data cleansing
    """

    df = raw_df.copy()
    df = df.replace(r"^\s*$", pd.NA, regex=True)

    datetime_utc_cols = ["datetime_utc", "publication_time_utc"]
    numeric_cols = ["price", "volume"]

    for dt_col in datetime_utc_cols:
        if dt_col in df.columns:
            df[dt_col] = pd.to_datetime(df[dt_col], unit="ns", errors="coerce", utc=True)

    for num_col in numeric_cols:
        if num_col in df.columns:
            df[num_col] = pd.to_numeric(df[num_col], errors="coerce")

    return df



In [ ]:
def construct_targets(
        df: pd.DataFrame
        ):
    """
    Construct quarter-hourly mFRR Up and Down activation targets.

    A period is classified as activated when realised activation energy is non-zero.
    """

    target_markets = {
        "mFRR Up Energy Activation": "mfrr_up_volume_mwh",
        "mFRR Down Energy Activation": "mfrr_down_volume_mwh"
    }

    target_data = df.loc[
        (df["market"].isin(target_markets)),
        [
            "datetime_utc",
            "market",
            "volume",
            "time_resolution"
        ]
    ].copy()

    invalid_resolution = (
        target_data["time_resolution"] != "15min"
    )

    if invalid_resolution.any():
        raise ValueError(
            "Non-15-minute observations found in mFRR activation data."
        )

    # There should be one observation per market and delivery period
    duplicates = target_data.duplicated(
        subset=["datetime_utc", "market"],
        keep=False
    )

    if duplicates.any():
        raise ValueError(
            "Duplicate activation observations found for a delivery period."
        )

    targets = (
        target_data
        .pivot(
            index="datetime_utc",
            columns="market",
            values="volume"
        )
        .rename(columns=target_markets)
        .reset_index()
    )

    for direction in ["up", "down"]:

        volume_col = f"mfrr_{direction}_volume_mwh"
        target_col = f"mfrr_{direction}_activated"

        targets[target_col] = pd.Series(
            pd.NA,
            index=targets.index,
            dtype="Int64"
        )

        available = targets[volume_col].notna()

        targets.loc[available, target_col] = (
            targets.loc[available, volume_col]
            .ne(0)
            .astype(int)
        )

    return (
        targets
        .sort_values("datetime_utc")
        .reset_index(drop=True)
    )

In [ ]:
def build_point_in_time_features(
        df: pd.DataFrame,
        forecast_origin: str,
        horizon_hours: int = 24
        ):
    """Retain only information published on or before the forecast origin to ensure no data leakage

    Args: 
        df (pd.DataFrame): cleaned long-format market data.

        forecast_origin (str): time at which the forecast is assumed to be made.

        horizon_hours (int): forecast horizon in hours.

        market_area (str): market area being modelled.

    Returns:
        pd.DataFrame: one row per future quarter-hour containing only information available at the forecast origin.
    """

    origin = pd.Timestamp(forecast_origin)

    if origin.tzinfo is None:
        origin = origin.tz_localize("UTC")
    else:
        origin = origin.tz_convert("UTC")

    n_periods = horizon_hours * 4

    features = pd.DataFrame({
        "datetime_utc": pd.date_range(
            start=origin + pd.Timedelta(minutes=15),
            periods=n_periods,
            freq="15min"
        )
    })

    features["forecast_origin_utc"] = origin
    features["horizon_qh"] = np.arange(1, n_periods + 1)

    # Build out calendar features 
    features["hour"] = features["datetime_utc"].dt.hour
    features["quarter_hour"] = features["datetime_utc"].dt.minute // 15
    features["day_of_week"] = features["datetime_utc"].dt.dayofweek
    features["is_weekend"] = (features["day_of_week"] >= 5).astype(int)
    features["_capacity_hour"] = features["datetime_utc"].dt.floor("h")


    capacity_markets = {
        "mFRR Up Capacity": "mfrr_up",
        "mFRR Down Capacity": "mfrr_down"
    }

    capacity = df.loc[(df["market"].isin(capacity_markets))].copy()

    # ensure no data-leakage
    capacity = capacity.loc[
        capacity["publication_time_utc"].notna()
        & (
            capacity["publication_time_utc"]
            <= origin
        )
    ]

    # Check expected product structure
    if not capacity.empty:

        invalid_resolution = (
            capacity["time_resolution"]
            .dropna()
            .ne("1h")
            .any()
        )

        if invalid_resolution:
            raise ValueError(
                "Unexpected time resolution found "
                "in mFRR capacity data."
            )

        invalid_volume_unit = (
            capacity["volume_unit"]
            .dropna()
            .ne("MW")
            .any()
        )

        if invalid_volume_unit:
            raise ValueError(
                "Unexpected volume unit found "
                "in mFRR capacity data."
            )

    # If multiple versions exist, retain the latest version
    capacity = (
        capacity
        .sort_values("publication_time_utc")
        .drop_duplicates(
            subset=["market", "datetime_utc"],
            keep="last"
        )
    )

    # Add the up/down capacity
    for market, prefix in capacity_markets.items():

        market_data = capacity.loc[
            capacity["market"] == market,
            [
                "datetime_utc",
                "price",
                "volume"
            ]
        ].copy()

        market_data = market_data.rename(
            columns={
                "datetime_utc": "_capacity_hour",
                "price": f"{prefix}_capacity_price_eur_mw",
                "volume": f"{prefix}_capacity_mw"
            }
        )

        features = features.merge(
            market_data,
            on="_capacity_hour",
            how="left"
        )

    features = features.drop(
        columns="_capacity_hour"
    )

    return features

In [ ]:
def add_historical_state_features(
        df: pd.DataFrame,
        features: pd.DataFrame,
        forecast_origin: str,
        lookback_periods: int = 96
        ):
    """Add historical market-state features that were genuinely available at the forecast origin.

    Args:
        df (pd.DataFrame): cleaned long-format market data.

        features (pd.DataFrame): point-in-time feature table for the forecast horizon.

        forecast_origin (str): timestamp at which the forecast is assumed to be made.

        lookback_periods (int, optional): Number of most recent observations used for rolling summaries.
        96 quarter-hours corresponds to 24 hours for 15-minute series. Defaults to 96.

    Returns:
        pd.DataFrame: feature table with historical state variables added.
    """


    features = features.copy()

    origin = pd.Timestamp(forecast_origin)

    if origin.tzinfo is None:
        origin = origin.tz_localize("UTC")
    else:
        origin = origin.tz_convert("UTC")

    # Only use information available at forecast time to avoid data leakage
    known = df.loc[
        (df["publication_time_utc"].notna())
        & (df["publication_time_utc"] <= origin)
        & (df["datetime_utc"] < origin)
    ].copy()

    # If observations were revised, retain the latest version
    known = (
        known
        .sort_values("publication_time_utc")
        .drop_duplicates(
            subset=["market", "datetime_utc"],
            keep="last"
        )
    )


    # imbalance prices
    imbalance = (
        known.loc[
            (known["market"] == "Imbalance Single")
            & (known["time_resolution"] == "15min")
        ]
        .sort_values("datetime_utc")
        .tail(lookback_periods)
    )

    if not imbalance.empty:

        features["imbalance_price_last"] = imbalance["price"].iloc[-1]
        features["imbalance_price_mean"] = imbalance["price"].mean()
        features["imbalance_price_std"] = imbalance["price"].std()
        features["imbalance_price_min"] = imbalance["price"].min()
        features["imbalance_price_max"] = imbalance["price"].max()

    # intraday close price
    intraday = (
        known.loc[
            (known["market"] == "Intraday Continuous Close Price")
            & (known["time_resolution"] == "15min")
        ]
        .sort_values("datetime_utc")
        .tail(lookback_periods)
    )

    if not intraday.empty:

        features["intraday_close_last"] = intraday["price"].iloc[-1]
        features["intraday_close_mean"] = intraday["price"].mean()
        features["intraday_close_std"] = intraday["price"].std()

   # historical mFRR activity 
    activation_markets = {
        "mFRR Up Energy Activation": "up",
        "mFRR Down Energy Activation": "down"
    }

    for market, direction in activation_markets.items():

        activation = (
            known.loc[
                (known["market"] == market)
                & (known["time_resolution"] == "15min")
            ]
            .sort_values("datetime_utc")
            .tail(lookback_periods)
        )

        if activation.empty:
            continue

        activated = activation["volume"].ne(0)

        features[f"mfrr_{direction}_activation_rate"] = activated.mean()
        features[f"mfrr_{direction}_volume_mean_mwh"] = activation["volume"].mean()
        features[f"mfrr_{direction}_volume_max_mwh"] = activation["volume"].max()

    return features

In [ ]:
def build_training_dataset(
        df: pd.DataFrame,
        targets: pd.DataFrame,
        start_origin: str,
        end_origin: str,
        forecast_hour: int = 12,
        horizon_hours: int = 24
        ):
    """Construct a historical point-in-time training dataset by repeatedly simulating a 24-hour forecast from daily forecast origins.
    

    Args:
        df (pd.DataFrame): cleaned market data.

        targets (pd.DataFrame):realised mFRR Up and Down activation targets.

        start_origin (str): first date used as a forecast origin.

        end_origin (str): final date used as a forecast origin.

        forecast_hour (int, optional): UTC hour at which the daily forecast is assumed to be made. Defaults to 12.

        horizon_hours (int, optional): Length of each forecast horizon. Defaults to 24.

    Returns:
        pd.DataFrame: historical point-in-time feature and target dataset.
    """

    start = pd.Timestamp(start_origin)

    if start.tzinfo is None:
        start = start.tz_localize("UTC")
    else:
        start = start.tz_convert("UTC")

    end = pd.Timestamp(end_origin)

    if end.tzinfo is None:
        end = end.tz_localize("UTC")
    else:
        end = end.tz_convert("UTC")

    origins = pd.date_range(
        start=start.normalize(),
        end=end.normalize(),
        freq="D",
        tz="UTC"
    ) + pd.Timedelta(hours=forecast_hour)

    datasets = []

    for origin in origins:

        features = build_point_in_time_features(
            df,
            forecast_origin=origin,
            horizon_hours=horizon_hours
        )

        features = add_historical_state_features(
            df,
            features,
            forecast_origin=origin
        )

        origin_data = features.merge(
            targets,
            on="datetime_utc",
            how="left"
        )

        datasets.append(origin_data)

    training_data = pd.concat(
        datasets,
        ignore_index=True
    )

    return training_data

In [29]:
def train_activation_models(training_data: pd.DataFrame):
    """
    Train separate XGBoost classifiers for mFRR Up and Down activation.
    """

    # Columns that must NEVER be predictors
    excluded_cols = [
        "datetime_utc",
        "forecast_origin_utc",

        # realised future information
        "mfrr_up_volume_mwh",
        "mfrr_down_volume_mwh",

        # targets
        "mfrr_up_activated",
        "mfrr_down_activated"
    ]

    feature_cols = [
        col for col in training_data.columns
        if col not in excluded_cols
    ]

    # Only use rows where both targets are available
    model_data = training_data.dropna(
        subset=[
            "mfrr_up_activated",
            "mfrr_down_activated"
        ]
    ).copy()

    X = model_data[feature_cols]

    models = {}

    for direction in ["up", "down"]:

        y = model_data[f"mfrr_{direction}_activated"].astype(int)

        n_positive = (y == 1).sum()
        n_negative = (y == 0).sum()

        scale_pos_weight = (
            n_negative / n_positive
            if n_positive > 0
            else 1
        )

        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            random_state=42
        )

        model.fit(X, y)

        models[direction] = model

    return models, feature_cols

In [ ]:
def predict_activation(
        df: pd.DataFrame,
        models: dict,
        feature_cols: list,
        forecast_origin: str
        ):
    """
    Forecast mFRR Up and Down activation probabilities
    for the next 24 hours.
    """

    features = build_point_in_time_features(
        df,
        forecast_origin=forecast_origin
    )

    features = add_historical_state_features(
        df,
        features,
        forecast_origin=forecast_origin
    )

    X = features[feature_cols]

    predictions = features[
        [
            "datetime_utc",
            "forecast_origin_utc",
            "horizon_qh"
        ]
    ].copy()

    predictions["p_mfrr_up_activation"] = (
        models["up"].predict_proba(X)[:, 1]
    )

    predictions["p_mfrr_down_activation"] = (
        models["down"].predict_proba(X)[:, 1]
    )

    return predictions

In [31]:
def main():

    raw_data = fetch_market_history(
        start_date="2026-03-01T00:00:00Z",
        end_date="2026-03-08T00:00:00Z"
    )

    cleaned = clean_market_data(raw_data)

    targets = construct_targets(cleaned)

    training_data = build_training_dataset(
        cleaned,
        targets,
        start_origin="2026-03-03",
        end_origin="2026-03-07",
        forecast_hour=12
    )

    models, feature_cols = train_activation_models(
        training_data
    )

    predictions = predict_activation(
        cleaned,
        models,
        feature_cols,
        forecast_origin="2026-03-08T12:00:00Z"
    )

    display(predictions)


if __name__ == "__main__":
    main()

2026-09-28 21:47:41,190 | INFO | Requesting market data from 2026-03-01T00:00:00+00:00 to 2026-03-01T06:00:00+00:00
2026-09-28 21:47:49,865 | INFO | Requesting market data from 2026-03-01T06:00:00+00:00 to 2026-03-01T12:00:00+00:00
2026-09-28 21:47:55,797 | INFO | Requesting market data from 2026-03-01T12:00:00+00:00 to 2026-03-01T18:00:00+00:00
2026-09-28 21:48:01,795 | INFO | Requesting market data from 2026-03-01T18:00:00+00:00 to 2026-03-02T00:00:00+00:00
2026-09-28 21:48:08,164 | INFO | Requesting market data from 2026-03-02T00:00:00+00:00 to 2026-03-02T06:00:00+00:00
2026-09-28 21:48:13,913 | INFO | Requesting market data from 2026-03-02T06:00:00+00:00 to 2026-03-02T12:00:00+00:00
2026-09-28 21:48:19,718 | INFO | Requesting market data from 2026-03-02T12:00:00+00:00 to 2026-03-02T18:00:00+00:00
2026-09-28 21:48:25,182 | INFO | Requesting market data from 2026-03-02T18:00:00+00:00 to 2026-03-03T00:00:00+00:00
2026-09-28 21:48:31,812 | INFO | Requesting market data from 2026-03-03T

,datetime_utc,forecast_origin_utc,horizon_qh,p_mfrr_up_activation,p_mfrr_down_activation
0,2026-03-08 12:15:00+00:00,2026-03-08 12:00:00+00:00,1,0.729551,0.184330
1,2026-03-08 12:30:00+00:00,2026-03-08 12:00:00+00:00,2,0.674193,0.230265
2,2026-03-08 12:45:00+00:00,2026-03-08 12:00:00+00:00,3,0.393638,0.372701
3,2026-03-08 13:00:00+00:00,2026-03-08 12:00:00+00:00,4,0.788295,0.117543
4,2026-03-08 13:15:00+00:00,2026-03-08 12:00:00+00:00,5,0.826202,0.160273
...,...,...,...,...,...
91,2026-03-09 11:00:00+00:00,2026-03-08 12:00:00+00:00,92,0.221514,0.215713
92,2026-03-09 11:15:00+00:00,2026-03-08 12:00:00+00:00,93,0.101982,0.236946
93,2026-03-09 11:30:00+00:00,2026-03-08 12:00:00+00:00,94,0.123525,0.255391
94,2026-03-09 11:45:00+00:00,2026-03-08 12:00:00+00:00,95,0.026877,0.182083
